In [ ]:
import os
import sys
import math
from pathlib import Path
import numpy as np
import random

# Importaciones de terceros
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from compressai.zoo import cheng2020_anchor

from torch.utils.tensorboard import SummaryWriter
import albumentations as A
from albumentations.pytorch import ToTensorV2
import optuna

In [ ]:
# Semilla para reproducibilidad de los experimentos
random.seed(42)
np.random.seed(42)
torch.manual_seed(42);

In [ ]:
# Dispositivo
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

In [ ]:
# Me aseguro de que el directorio raíz del proyecto esté en el sys.path
project_root = Path(os.path.abspath("")).parent
if project_root not in sys.path:
    sys.path.append(str(project_root))

In [ ]:
# Importo las funciones de configuración
from src.config import processed_data_dir, models_dir, reports_dir, load_config
from src.utils.datasets import CustomDataset_2_2
from src.models.compressai_chang2020_model.train_model import train_model_optuna

In [ ]:
# Cargo la configuración 
config = load_config()

In [ ]:
# Transformación de las imágenes
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

In [ ]:
# Definir el pipeline de augmentación
augmentation_pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.OneOf([
        A.Rotate(limit=(0, 0), p=0.33),
        A.Rotate(limit=(180, 180), p=0.33),
        A.Compose([A.Rotate(limit=(180, 180), p=1.0), A.HorizontalFlip(p=1.0)], p=0.34)
    ], p=1.0)
])

In [ ]:
# Ruta de los datos
final_data_dir = processed_data_dir()

In [ ]:
# Crear los datasets
train_dataset = CustomDataset_2_2(
    csv_file=os.path.join(final_data_dir, 'train_2.csv'),
    transform=transform,
    augmentation_pipeline=augmentation_pipeline,
    use_augmentation=True  # Activar Data Augmentation para el entrenamiento y probar con imágenes aumentadas
)

val_dataset = CustomDataset_2_2(
    csv_file=os.path.join(final_data_dir, 'val_2.csv'),
    transform=transform,
    use_augmentation=False  # No activar Data Augmentation para el conjunto de validación
)

In [ ]:
def custom_loss(output, target, bit_rate, lambda_value):
    mse_loss = nn.MSELoss()(output, target)
    distortion_loss = 255**2 * mse_loss
    
    # Si bit_rate es un escalar, no se necesita calcular la media
    rate_loss = bit_rate if isinstance(bit_rate, float) else bit_rate.mean()
    
    loss = lambda_value * distortion_loss + rate_loss
    return loss


In [ ]:
# Definir la función objetivo para Optuna
def objective(trial):
    # Sugerir hiperparámetros
    learning_rate = trial.suggest_float('learning_rate', 1e-6, 1e-5, log=True)
    batch_size = trial.suggest_categorical('batch_size', [8, 16, 32])
    lambda_value = trial.suggest_float('lambda', 1e-2, 1, log=True)
    
    # Modelo y optimizador
    model = cheng2020_anchor(quality=1, pretrained=True).to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    output_model_path = models_dir() / "trained" / "cheng_optuna_trials_PPT" 
    os.makedirs(output_model_path, exist_ok=True)
    
    # Redefinir DataLoader con el batch_size sugerido
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    
    # Scheduler
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)
    
    # Definir directorio base de logs para tensorboard
    log_base_dir = reports_dir() / 'logs' / 'cheng_optuna_trials_PPT'
    os.makedirs(log_base_dir, exist_ok=True)

    # Definir subdirectorio de logs para el trial de Optuna
    log_dir = log_base_dir / f'optuna_trial_{trial.number}'
    os.makedirs(log_dir, exist_ok=True)

    # Inicializar tensorboard para el trial
    writer = SummaryWriter(log_dir=str(log_dir), comment=f"Cheng_Optuna_Trial_{trial.number}")

    # Entrenar el modelo (prueba con pocas épocas para la búsqueda de hiperparámetros)
    num_epochs = 10  # Puedes ajustar esto para pruebas rápidas
    train_losses, val_losses, val_psnr_values, val_ssim_values, compression_ratios = (
        train_model_optuna(
            model,
            train_loader,
            val_loader,
            custom_loss,  # Aquí pasamos la función de pérdida
            optimizer,
            scheduler,
            output_model_path, 
            nombre_modelo=f'optuna_trial_{trial.number}',
            num_epochs=num_epochs,
            device=device,
            writer=writer,  # Usar TensorBoard para monitorear las métricas
            lambda_value=lambda_value
        )
    )
    print()  # Línea en blanco

    writer.close()  # Cerrar el writer de TensorBoard para este trial
    
    # Optuna minimiza la función objetivo, por lo que devolvemos la última pérdida de validación
    return val_losses[-1]

In [ ]:
%load_ext tensorboard

%tensorboard --logdir=../reports/logs/cheng_optuna_trials_PPT --host 0.0.0.0 --port 6006

In [ ]:
# Crear un estudio de Optuna y optimizar
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30)

In [ ]:
# Imprimir los mejores hiperparámetros
print("Best hyperparameters: ", study.best_params)
print("Best validation loss: ", study.best_value)